# <font color="red"> **3.3 Análisis de Extremos**

En esta clase, continuaremos viendo otras formas de identificar extremos.

In [ ]:
#Vamos a llamar a las librerías necesarias 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

------

## <font color="blueviolet">**Rango intercuartílico (IQR)**

El `Rango Intercuartílico (IQR)` es una medida de dispersión que permite identificar valores extremos (outliers) comparando los datos con la distribución central. Para  calcularlo usamos la fórmula:

$$
IQR=Q3-Q1
$$

donde $Q1$ representa el percentil 25 ($P_{25}$) y $Q3$ el percentil 75 ($P_{75}$).

#### <font color="steelblue">**¿Cómo detectamos valores extremos?**

Un valor se considera extremo si:

$$
x<Q1-1.5*IQR
$$
$$
x>Q3+1.5*IQR
$$

Es un método robusto porque no depende de un solo valor (como el máximo o mínimo) y no se ve afectado fácilmente por outliers. **Un “extremo” según IQR significa: “Este valor es inusualmente alto o bajo respecto al comportamiento típico de la serie.”**

#### <font color="orange"> **Ejemplo 1**

In [132]:
#Vamos a encontrar los valores extremos de nuestros datos de gasolina
gas=pd.read_csv("gas_prices.csv")
gas_mxn=gas.iloc[:, 1:] = gas.iloc[:, 1:].mul(18).round(2) #.mul(), puedo multiplicar el dataframe por un valor
gas_mxn.head()

,Australia,Canada,France,Germany,Italy,Japan,Mexico,South Korea,UK,USA
0,NaN,33.66,65.34,47.70,82.62,56.88,18.00,36.90,50.76,20.88
1,35.28,34.56,62.10,52.20,81.00,62.28,23.40,44.82,54.18,20.52
2,34.02,31.14,64.08,58.86,81.54,64.44,27.00,47.70,55.08,20.34
3,31.14,28.26,61.38,55.26,66.24,74.88,28.08,51.84,51.12,19.98
4,33.12,26.10,64.62,63.36,66.60,78.48,26.64,51.66,53.82,19.98


In [133]:
#Primero calculamos el rango intercuartílico

#******************Q1 y Q2*****************
Q1=np.nanpercentile(gas_mxn,[25])
Q3=np.nanpercentile(gas_mxn,[75])

#Rango intercuartílico
IQR=Q3-Q1

In [134]:
#Vamos a detectar valores inusualmente altos y bajos
limite_superior= Q3+(1.5*IQR)
limite_inferior= Q1-(1.5*IQR)

In [135]:
#Vamos a ver los resultados

(array([132.93]), array([-23.31]))

In [136]:
#Puedo ir checando valor por valor de mi df si es anómalo o no

#Vamos a ver si el valor que se encuentra en esta posición gas_mxn.iloc[18,4] es anómalo o nó
if  gas_mxn.iloc[18,4]> limite_superior:
    print("Anómalamente alto")
elif np.array(gas_mxn.iloc[18,4])<limite_inferior:
    print ("Anómalamente bajo")

else:
    print("No es anómalo")

Anómalamente alto


In [137]:
#Así como hacíamos en los percentiles podemos colorear nuestro df en relación a si el valor es anómalo o no

#Definimos nuestra función

def color_fondo(value, column, perc_gas):
    #Para los anómalos bajos
    if value < perc_gas[0]:
        color = 'steelblue'

    #Para anómalos altos
    elif value > perc_gas[1]:
        color = 'salmon'
    else:
        color = 'None' #Los deja en blanco
    return f'background-color: {color}'

#Aplicando una función lambda
color_change=lambda x: x.apply(color_fondo, column = x.name, perc_gas = [limite_inferior,limite_superior])

gas_mxn.style.apply(color_change, axis = 0).format(precision=2)

,Australia,Canada,France,Germany,Italy,Japan,Mexico,South Korea,UK,USA
0,nan,33.66,65.34,47.70,82.62,56.88,18.00,36.90,50.76,20.88
1,35.28,34.56,62.10,52.20,81.00,62.28,23.40,44.82,54.18,20.52
2,34.02,31.14,64.08,58.86,81.54,64.44,27.00,47.70,55.08,20.34
3,31.14,28.26,61.38,55.26,66.24,74.88,28.08,51.84,51.12,19.98
4,33.12,26.10,64.62,63.36,66.60,78.48,26.64,51.66,53.82,19.98
5,35.10,27.54,76.68,71.28,72.00,79.74,19.98,52.92,57.78,20.70
6,38.16,28.98,79.38,70.92,79.02,65.52,22.50,57.24,60.12,22.14
7,36.90,29.16,72.00,63.54,73.26,58.68,26.46,60.12,68.94,22.14
8,29.34,24.84,69.66,60.12,69.12,50.76,26.82,54.72,73.08,19.08
9,30.96,27.36,69.30,61.56,69.66,58.86,32.22,68.40,77.22,21.06


**¿Es correcto lo que estamos  obteniendo?, Vamos a comprobarlo**

In [ ]:
# Vamos a hacer un boxplot

#Si observan, tiene un nan en Australia. Podríamos aplicar un dropna pero eliminaríamos toda la fila
#Vamos a quitar los nan por columnas
data = [gas_mxn[col].dropna() for col in gas_mxn.columns]

In [ ]:
#Punto2
paises=gas_mxn.columns
plt.boxplot(data, tick_labels=paises,vert=True,showmeans=True,
           showfliers=True,  
           patch_artist=True, 
            meanline=True,
            boxprops = dict(facecolor= 'darkturquoise'),
            medianprops = dict(color = "darkgoldenrod", linewidth = 1.5), 
            meanprops = dict(color = "red", linewidth = 1.2),
            whiskerprops = dict(color = "purple", linewidth = 2),
            capprops = dict(color = "maroon", linewidth = 2.5), #Personalizar extremos
            flierprops = dict(marker = '*', markersize = 10, markerfacecolor = 'darkolivegreen')#Personalizar outliers
           ) 
plt.title("Con outliers")
plt.xticks(rotation=45)
plt.ylabel("Precios gas")
plt.show()

#¿Nuestros resultados son consistentes?

`**IMPORTANTE**: Si mezclamos datos con comportamientos distintos, el IQR deja de detectar extremos porque interpreta esa variabilidad como normal. El IQR solo funciona bien cuando los datos vienen de la misma distribución.`

En este caso, si mezclamos países con distintos niveles de precios, el IQR crece artificialmente y deja de detectar valores extremos. Para solucionarlo, podríamos analizar cada distribución por separado, es decir, por grupo homogéneo (país, año).

In [ ]:
#Ahora vamos a calcular los rangos intercuartiles por país

lim_s={}
lim_i={}
for i in gas_mxn.columns:
    q1=np.nanpercentile(gas_mxn[i],[25])
    q3=np.nanpercentile(gas_mxn[i],[75])
    iqr=q3-q1
    limite_s= q3+(1.5*iqr)
    limite_i= q1-(1.5*iqr)
    lim_s[i]=limite_s
    lim_i[i]=limite_i

In [ ]:
#Vamos a crear nuestra función
def color_fondo(value, column):
    
    if pd.isna(value):
        return ''  # no colorear NaN
    
    if value < lim_i[column]:
        color = 'steelblue'
    elif value > lim_s[column]:
        color = 'salmon'
    else:
        color = ''
        
    return f'background-color: {color}'

In [ ]:
#Primero tomamos una columna, luego evaluamos cada valor dentro de su propio contexto, y 
#finalmente pintamos según si es extremo o no.
color_change = lambda x: x.apply(lambda v: color_fondo(v, x.name)) 

gas_mxn.style.apply(color_change, axis=0).format(precision=2)

#¿Qué podemos observar?

`CUIDADO: Si los datos tienen tendencia o mezclan contextos, el método pierde sensibilidad`

#### <font color="orange"> **Ejemplo 2**

In [ ]:
#Vamos a analizar datos más aleatorios
titanic=pd.read_csv("titanic.csv")
titanic_slice=titanic[["Age"]]

In [ ]:
#Vamos a obtener el IQR

#******************Q1 y Q3*****************
Q1_age=np.nanpercentile(titanic_slice,[25])
Q3_age=np.nanpercentile(titanic_slice,[75])

#Rango intercuartílico
IQR_age=Q3_age-Q1_age

In [ ]:
ls_age= Q3_age+(1.5*IQR_age)
li_age= Q1_age-(1.5*IQR_age)

In [ ]:
def color_fondo(value, column, perc_edad):
    #Para nos anómalos bajos
    if value < perc_edad[0]:
        color = 'steelblue'

    #Para los anómalos altos
    elif value > perc_edad[1]:
        color = 'salmon'
    else:
        color = 'None' #Los deja en blanco
    return f'background-color: {color}'

#Aplicando una función lambda
color_change=lambda x: x.apply(color_fondo, column = x.name, perc_edad = [li_age,ls_age])

#titanic_slice.style.apply(color_change, axis = 0).format(precision=2)

In [ ]:
#Vamos a observarlo gráficamente

plt.boxplot(titanic_slice, tick_labels=["Age"],vert=True,showmeans=True,
           showfliers=True,  
           patch_artist=True, 
            meanline=True,
            boxprops = dict(facecolor= 'darkturquoise'),
            medianprops = dict(color = "darkgoldenrod", linewidth = 1.5), 
            meanprops = dict(color = "red", linewidth = 1.2),
            whiskerprops = dict(color = "purple", linewidth = 2),
            capprops = dict(color = "maroon", linewidth = 2.5), #Personalizar extremos
            flierprops = dict(marker = '*', markersize = 10, markerfacecolor = 'darkolivegreen')#Personalizar outliers
           ) 
plt.xticks(rotation=45)
plt.ylabel("Edades")
plt.show()


El IQR puede detectar extremos severos, pero no está diseñado específicamente para diferenciarlos si son extremos moderados o severos. 

Pueden usar otra escala para detectar extremos:  $3 *$ $IQR$. Sin embargo, debido a que el IQR se calcula con $Q_{1}$ y $Q_{3}$ puede que no vea bien los extremos y a los subestime.


#### <font color="brown"> Rol real del IQR en ciencias de la Tierra:

El IQR se usa principalmente para:

1. Control de calidad
2. Análisis exploratorio (EDA)

**Por lo que no es el método principal para extremos en CT**

-----------------------------------------------------------

## <font color="blueviolet"> **Máximos y Mínimos**

Los `máximos` y `mínimos` como análisis de extremos consisten en identificar los valores más altos y más bajos de una variable para detectar eventos extremos absolutos dentro de un periodo de estudio (describe un solo punto de la distribución). Son una forma de análisis de extremos basada en valores absolutos, no en probabilidades ni umbrales estadísticos. Se consideran una medida de extremos porque:

 * Representan los límites observados del sistema
 * Permiten analizar la intensidad máxima alcanzada

<font color="brown">**Algunos usos en CT**

|Climatología|Hidrología|Criósfera| Espaciales|Tierra Sólida|
|------------|-------------|----------|----|-----|
 |Temperatura máxima diaria |Precipitación máxima diaria|Espesor máximo/mínimo de hielo|Cantidad máxima de meteoritos que cayeron en el año|Intensidad máxima de los sismos registrados en un día| 

 
<font color ="brown">**Limitaciones**

* Sensibles a valores atípicos: un solo dato puede dominar el resultado.

* No representan la distribución. no dicen si el valor es común, raro, o extremo relativo.

* Dependencia del tamaño de muestra: Más datos → mayor probabilidad de valores extremos.

* No capturan frecuencia: solo describen un evento, no cuántas veces ocurre.

* Sensibles a no estacionariedad: en series con tendencia, los máximos aumentan aunque no haya eventos extremos reales.

#### <font color="orange"> **Ejemplo 1**

In [ ]:
#Vamos a trabajar con datos de hielo marino del ártico 
#(https://data.marine.copernicus.eu/product/SEAICE_GLO_PHY_CLIMATE_L3_MY_011_013/description)

In [ ]:
data=xr.open_dataset("artic_sea_ice_thickness.nc") #¿Cómo es la frecuencia?
sea_ice=data.sea_ice_thickness 

In [ ]:
#Vamos a elegir los años 2020 y 2021

sea_ice_sliced=sea_ice.where((sea_ice.time.dt.year>2019)&(sea_ice.time.dt.year<2022),drop=True)

#Vamos a obtener el máximo y mínimo registrado de cada año
maximo_2020=sea_ice_sliced.where((sea_ice.time.dt.year==2020),drop=True).max()  
maximo_2021=sea_ice_sliced.where((sea_ice.time.dt.year==2021),drop=True).max()

minimo_2020=sea_ice_sliced.where((sea_ice.time.dt.year==2020),drop=True).min() 
minimo_2021=sea_ice_sliced.where((sea_ice.time.dt.year==2021),drop=True).min()

In [ ]:
print("2020:", "máximo:", maximo_2020.values, "mínimo:", minimo_2020.values)
print("2020:" "máximo:", maximo_2021.values, "mínimo:", minimo_2021.values)

In [ ]:
#¿En qué mes se registraron los máximos?

#### <font color="orange"> **Ejemplo 2: Distribución espacial**

In [ ]:
maximo_2020_s=sea_ice_sliced.where((sea_ice.time.dt.year==2020),drop=True).max("time")  
maximo_2021_s=sea_ice_sliced.where((sea_ice.time.dt.year==2021),drop=True).max("time")

minimo_2020_s=sea_ice_sliced.where((sea_ice.time.dt.year==2020),drop=True).min("time") 
minimo_2021_s=sea_ice_sliced.where((sea_ice.time.dt.year==2021),drop=True).min("time")

In [ ]:
#Supongamos que queremos el la distribución de los máximos en el 2021

#Coordenadas
lat=maximo_2021_s.latitude
lon=maximo_2021_s.longitude

ax = plt.axes(projection=ccrs.NorthPolarStereo())

# Limitar al Ártico
ax.set_extent([-180, 180, 60, 90], crs=ccrs.PlateCarree())

ax.coastlines()

gl = ax.gridlines(draw_labels=False, linestyle='--', alpha=0.5)

levels = np.linspace(float(maximo_2021_s.min()),float(maximo_2021_s.max()),15)

cs = ax.contourf(lon, lat, maximo_2021_s,levels=levels,cmap="RdBu_r",extend="both",transform=ccrs.PlateCarree())

cbar = plt.colorbar(cs, pad=0.05)
cbar.set_label('[m]')

plt.title('Valores máximos 2021', fontsize=14)

plt.show()

# Los valores obtenidos son raros?, extremos?

**En resumen, los máximos y mínimos muestran los límites observados, pero no indican qué tan raros o representativos son dentro de la distribución.**

### <font color="purple"> Ejercicio 1:

A partir de los datos `Mexico_ppt.nc`:

* Encuentra los máximos y mínimos de precipitación para cada año del periodo 1990-2022. Gráfica tus resultados en una solo panel.
* Encuentra la distribución espacial de los máximos y mínimos de precipitación para el año 2022.

-------------------
## <font color="blueviolet"> Índices climáticos extremos
Los `índices climáticos` de extremos son métricas estandarizadas que cuantifican la frecuencia, intensidad y duración de eventos extremos en variables climáticas. Permiten transformar datos diarios en indicadores comparables de extremos. La mayoría de estos índices fueron definidos por Expert Team on Climate Change Detection and Indices (ETCCDI). 

Para saber más puedes visitar los siguientes links:
* <https://www.climdex.org/learn/indices/>
* <https://www.clivar.org/clivar-panels/etccdi/indices-data/indices-data>
------------

## <font color="blueviolet"> Extreme Value Theory (EVT)

La Extreme Value Theory (EVT) es una rama de la estadística que modela el comportamiento de los valores más extremos de una variable. Se enfoca en las colas de la distribución, donde ocurren los eventos raros.

* <https://www.sciencedirect.com/topics/mathematics/extreme-value-theory>
* <https://hess.copernicus.org/preprints/hess-2016-65/hess-2016-65.pdf>
* <https://grotjahn.ucdavis.edu/EWEs/extremes_primer_v9_22_15.pdf>
* <https://etrp.wmo.int/pluginfile.php/68830/mod_resource/content/1/6.-%20Valores%20Extremos%20P.Retorno.pdf>

--------------------------------------------

### **Resumen de métodos para identificar extremos**

| Método | Qué mide | Cómo se define | Tipo de extremo | Ventajas | Limitaciones | Uso típico |
|------|---------|---------------|----------------|----------|-------------|-----------|
| **Percentiles (P90, P95, P99)** | Posición relativa en la distribución | X > P95 o X < P5 | Relativo | ✔ Estándar en climatología <br> ✔ Comparable entre regiones | ❌ Depende del periodo base | Índices climáticos, extremos térmicos y de precipitación |
| **Anomalías estandarizadas** | Desviación respecto a la media | Z = (X - μ) / σ | Relativo | ✔ Considera media y variabilidad <br> ✔ Útil para comparar regiones | ❌ Asume cierta normalidad | Sequías, calor extremo, variabilidad climática |
| **Máximos y mínimos** | Valores más extremos observados | max(X), min(X) | Absoluto | ✔ Muy simple <br> ✔ Detecta récords | ❌ Sensible a outliers <br> ❌ No mide frecuencia | Récords climáticos, límites físicos |
| **IQR (rango intercuartílico)** | Dispersión central | IQR = Q3 - Q1 <br> Extremos: X > Q3 + 1.5*IQR | Relativo (outliers) | ✔ Robusto a outliers <br> ✔ No asume normalidad | ❌ No captura colas extremas <br> ❌ Sensible a tendencia | Control de calidad, análisis exploratorio |
| **Umbrales (thresholds)** | Valores que cruzan un límite | X > U | Absoluto o relativo | ✔ Interpretación física <br> ✔ Muy usado en impactos | ❌ Elección subjetiva del umbral | Olas de calor, lluvia extrema |
| **EVT (teoría de extremos)** | Comportamiento de eventos raros | GEV / GPD | Extremos severos | ✔ Modela colas <br> ✔ Estima periodos de retorno | ❌ Complejo <br> ❌ Requiere muchos datos | Riesgo climático, eventos raros |


Un extremo absoluto se define a partir de un valor fijo independiente del contexto (por ejemplo, temperatura > 35 °C), mientras que un extremo relativo depende de la distribución de los datos y se identifica en función de qué tan inusual es un valor dentro de su propio contexto (por ejemplo, valores por encima del percentil 95). En otras palabras, el extremo absoluto mide magnitudes físicas directamente comparables, mientras que el relativo evalúa rareza o desviación respecto al comportamiento típico.